In [1]:
"""
========================================================================
QA Script for FAST Results
========================================================================
For a sample of buildings from a FAST output CSV, this script recomputes
the loss values from the raw inputs and compares them to what FAST
reported. The goal is to confirm FAST's internal math is consistent.

What gets checked:
  1. Building Loss   = (BldgDmgPct / 100) × Cost
  2. Content Loss    = (ContDmgPct / 100) × ContentCostUSD
  3. Inventory Loss  = (InvDmgPct  / 100) × InventoryCostUSD

If any computed value disagrees with FAST's reported value by more
than TOLERANCE_USD, the row is flagged with pass or fail.
========================================================================
"""

"\n========================================================================\nQA Script for FAST Results\n========================================================================\nFor a sample of buildings from a FAST output CSV, this script recomputes\nthe loss values from the raw inputs and compares them to what FAST\nreported. The goal is to confirm FAST's internal math is consistent.\n\nWhat gets checked:\n  1. Building Loss   = (BldgDmgPct / 100) × Cost\n  2. Content Loss    = (ContDmgPct / 100) × ContentCostUSD\n  3. Inventory Loss  = (InvDmgPct  / 100) × InventoryCostUSD\n\nIf any computed value disagrees with FAST's reported value by more\nthan TOLERANCE_USD, the row is flagged with pass or fail.\n========================================================================\n"

## Imports

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
import getpass

## Config

In [3]:
# getpass.getuser() returns the current Windows username 
USER = getpass.getuser()

# Which FAST output CSV to QA. 
INPUT_CSV = Path(
    rf"C:\Users\{USER}\Pathways Dropbox\Projects\202501.Solano.Bayshore"
    rf"\400-Technical\450-FAST\CoastalA"
    rf"\NSI_Solano_FAST_Solano_inundation_rast_12.csv"
)

OUTPUT_CSV = Path(rf"C:\Users\{USER}\Pathways Dropbox\Projects\202501.Solano.Bayshore\400-Technical\450-FAST")

# How many buildings to audit
N_RANDOM  = 10   # random sample of buildings with loss > 0 (for representative spot-checks)
N_LARGEST = 5    # plus the N highest-loss buildings (audit the most consequential ones)

# How close is "close enough" — small rounding differences under $1 are ignored
TOLERANCE_USD = 1.00

## Main Script

In [4]:
# ========================================================================
# STEP 1 — Load the FAST output CSV
# ========================================================================
# pandas reads the whole file into memory. low_memory=False avoids dtype
# warnings when mixed types appear in the same column.
df = pd.read_csv(INPUT_CSV, low_memory=False)
print(f"Loaded {len(df):,} rows from {INPUT_CSV.name}\n")


# ========================================================================
# STEP 2 — Filter to buildings that actually had losses
# ========================================================================
# Buildings with $0 loss are not interesting to audit (0 = 0 is trivially correct).
# We only check buildings where FAST computed a non-zero building loss.
loss_rows = df[df["BldgLossUSD"] > 0].copy()
print(f"Rows with BldgLossUSD > 0: {len(loss_rows):,}\n")


# ========================================================================
# STEP 3 — Build the QA sample (random + top N by loss)
# ========================================================================
# Two-part sample to balance representativeness with auditing high-impact rows:
#   - Random sample gives a fair cross-section of buildings
#   - Top-N-by-loss makes sure we audit the rows that contribute most to the totals
# random_state=42 means the same 10 random rows get picked every run (reproducibility).
random_sample  = loss_rows.sample(n=min(N_RANDOM, len(loss_rows)), random_state=42)
largest_sample = loss_rows.nlargest(N_LARGEST, "BldgLossUSD")

# Combine the two samples, then drop any duplicates
# (a building might be both a random pick AND a top-N).
sample = pd.concat([random_sample, largest_sample]).drop_duplicates(subset="UserDefinedFltyId")
print(f"QA sample size: {len(sample)} buildings\n")


# ========================================================================
# STEP 4 — Recompute each loss and flag mismatches
# ========================================================================
# Empty list — append a result dict per audited building, then turn it into a DataFrame at the end.
results = []

# .iterrows() walks the sample row by row. We ignore the index (_) and
# get the row data as a pandas Series.
for _, row in sample.iterrows():

    # ---- Pull all the values we need from this row ----
    rid          = row["UserDefinedFltyId"]
    occ          = row["Occ"]
    cost         = row["Cost"] or 0                 # building replacement cost
    bldg_dmg     = row["BldgDmgPct"] or 0           # FAST's reported damage %
    bldg_loss    = row["BldgLossUSD"] or 0          # FAST's reported $ loss
    content_cost = row.get("ContentCostUSD", 0) or 0
    cont_dmg     = row.get("ContDmgPct", 0) or 0
    cont_loss    = row.get("ContentLossUSD", 0) or 0
    inv_cost     = row.get("InventoryCostUSD", 0) or 0
    inv_dmg      = row.get("InvDmgPct", 0) or 0
    inv_loss     = row.get("InventoryLossUSD", 0) or 0
    first_floor  = row["FirstFloorHt"]
    depth_struc  = row["Depth_in_Struc"]
    bddf_id      = row.get("BDDF_ID", "")

    # ---- CHECK 1: Building loss math ----
    # Formula: Loss = (DamagePct / 100) × Cost
    bldg_loss_computed = (bldg_dmg / 100.0) * cost
    bldg_diff          = bldg_loss_computed - bldg_loss
    bldg_flag = "FAIL" if abs(bldg_diff) > TOLERANCE_USD else "PASS"

    # ---- CHECK 2: Content loss math ----
    cont_loss_computed = (cont_dmg / 100.0) * content_cost
    cont_diff          = cont_loss_computed - cont_loss
    cont_flag = "FAIL" if abs(cont_diff) > TOLERANCE_USD else "PASS"

    # ---- CHECK 3: Inventory loss math ----
    inv_loss_computed  = (inv_dmg / 100.0) * inv_cost
    inv_diff           = inv_loss_computed - inv_loss
    inv_flag  = "FAIL" if abs(inv_diff)  > TOLERANCE_USD else "PASS"

    # ---- CHECK 4: Depth_in_Struc consistency (only if Depth_in_Grid exists) ----
    # Logic: Depth_in_Struc should equal max(Depth_in_Grid − FirstFloorHt, 0)
    # because the depth-damage curves use water height above the first floor,
    # not above the ground.
    depth_grid = row.get("Depth_in_Grid", np.nan)
    if pd.notna(depth_grid):
        expected_dis = max(depth_grid - first_floor, 0)
        dis_flag  = "FAIL" if abs(inv_diff)  > TOLERANCE_USD else "PASS"
    else:
        # Most FAST outputs don't include Depth_in_Grid, so this check is skipped
        expected_dis = np.nan
        dis_flag = "n/a"

    # ---- Save this building's results ----
    results.append({
        "UserDefinedFltyId":   rid,
        "Occ":                 occ,
        "Cost":                cost,
        "FirstFloorHt":        first_floor,
        "Depth_in_Struc":      depth_struc,
        # Building loss check
        "BldgDmgPct":          bldg_dmg,
        "BldgLoss_reported":   bldg_loss,
        "BldgLoss_computed":   round(bldg_loss_computed, 2),
        "Bldg_diff":           round(bldg_diff, 2),
        "Bldg_FLAG":           bldg_flag,
        # Content loss check
        "ContDmgPct":          cont_dmg,
        "ContLoss_reported":   cont_loss,
        "ContLoss_computed":   round(cont_loss_computed, 2),
        "Cont_diff":           round(cont_diff, 2),
        "Cont_FLAG":           cont_flag,
        # Inventory loss check
        "InvDmgPct":           inv_dmg,
        "InvLoss_reported":    inv_loss,
        "InvLoss_computed":    round(inv_loss_computed, 2),
        "Inv_diff":            round(inv_diff, 2),
        "Inv_FLAG":            inv_flag,
        # The BDDF_ID tells us which damage curve was used — useful for tracing
        # back to a specific curve if you want to audit the lookup itself
        "BDDF_ID":             bddf_id,
    })


# ========================================================================
# STEP 5 — Turn results into a DataFrame for nice printing/exporting
# ========================================================================
qa = pd.DataFrame(results)


# ========================================================================
# STEP 6 — Print the results in readable tables
# ========================================================================
print("=" * 100)
print("BUILDING LOSS QA — does (BldgDmgPct / 100) × Cost == BldgLossUSD?")
print("=" * 100)
print(qa[[
    "UserDefinedFltyId", "Occ", "Cost", "BldgDmgPct",
    "BldgLoss_reported", "BldgLoss_computed", "Bldg_diff", "Bldg_FLAG"
]].to_string(index=False))

print("\n" + "=" * 100)
print("CONTENT LOSS QA — does (ContDmgPct / 100) × ContentCostUSD == ContentLossUSD?")
print("=" * 100)
print(qa[[
    "UserDefinedFltyId", "ContDmgPct",
    "ContLoss_reported", "ContLoss_computed", "Cont_diff", "Cont_FLAG"
]].to_string(index=False))

# Count how many rows failed each check
n_bldg_flag = (qa["Bldg_FLAG"] == "FAIL").sum()
n_cont_flag = (qa["Cont_FLAG"] == "FAIL").sum()
n_inv_flag  = (qa["Inv_FLAG"]  == "FAIL").sum()

print(f"\nSummary: {len(qa)} buildings checked")
print(f"  Building loss math errors:  {n_bldg_flag}")
print(f"  Content loss math errors:   {n_cont_flag}")
print(f"  Inventory loss math errors: {n_inv_flag}")


# ========================================================================
# STEP 7 — Save the QA results next to the input CSV
# ========================================================================
# .stem strips the file extension; we prepend "QA_" so it's obvious what
# the file is. Saved in the same folder as the input.
out_path = OUTPUT_CSV / f"QA_{INPUT_CSV.stem}.csv"
qa.to_csv(out_path, index=False)
print(f"\nQA report saved to: {out_path}")


# ========================================================================
# STEP 8 — List the unique damage curves used in this sample
# ========================================================================
# If you want to go deeper and audit the actual depth-damage curves
# (not just the multiplication math), you'd need to open FAST's DDF
# lookup database and pull the (depth, %damage) pairs for these BDDF_IDs.
unique_bddfs = sorted(qa["BDDF_ID"].dropna().unique())
print(f"\nUnique BDDF_IDs in this QA sample: {unique_bddfs}")
print("To audit the actual curves, open FAST's damage function lookup file")
print(rf"(typically in C:\Users\{USER}\Documents\GitHub\FAST\src\)")
print("and pull the (depth, %damage) pairs for these IDs.")

Loaded 99,198 rows from NSI_Solano_FAST_Solano_inundation_rast_12.csv

Rows with BldgLossUSD > 0: 26

QA sample size: 14 buildings

BUILDING LOSS QA — does (BldgDmgPct / 100) × Cost == BldgLossUSD?
 UserDefinedFltyId   Occ         Cost  BldgDmgPct  BldgLoss_reported  BldgLoss_computed  Bldg_diff Bldg_FLAG
         473099535  RES2   64420.1305   15.519161        9997.443661            9997.46       0.02      PASS
         473100997  RES2   62344.5187   16.725397       10427.281574           10427.37       0.09      PASS
         472023675  COM4  344908.4240    9.311852       32117.322421           32117.36       0.04      PASS
         474168706 RES3A  634194.1650   15.493539       98259.096840           98259.12       0.03      PASS
         473100937  RES2   75017.6866   20.561514       15424.630883           15424.77       0.14      PASS
         473099536  RES2   59404.5460   33.264055       19760.179382           19760.36       0.18      PASS
         473100960  RES2   63907.1874  